In [1]:
# Cell 1: pinned versions (these were tested together)
!pip install -q transformers==5.22.1 trl==1.17.1 peft==0.21.2 bitsandbytes

In [2]:
# Cell 2: load the base model in 4-bit

os.environ["CUDA_VISIBLE_DEVICES"] = "0"  # Kaggle's T4 x2 exposes 2 GPUs; SFTTrainer needs exactly one

import glob, json, re, time
import os
import torch
from datasets import load_dataset
from peft import LoraConfig
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from trl import SFTConfig, SFTTrainer

BASE = "Qwen/Qwen2.5-Coder-1.5B-Instruct"
TRAIN = glob.glob("/kaggle/input/**/train.jsonl", recursive=True)[0]
SMOKE = glob.glob("/kaggle/input/**/smoke_prompts.jsonl", recursive=True)[0]
OUT = "/kaggle/working"

tokenizer = AutoTokenizer.from_pretrained(BASE)
model = AutoModelForCausalLM.from_pretrained(
    BASE,
    dtype=torch.float16,
    quantization_config=BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16),
    device_map={"": 0},
)
print(TRAIN, torch.cuda.get_device_name(0))


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

/kaggle/input/datasets/farhanziarizky/nl2sql-id-toy/train.jsonl Tesla T4


In [3]:
# Cell 3: train a LoRA adapter (loss only on the SQL)
trainer = SFTTrainer(
    model=model,
    train_dataset=load_dataset("json", data_files=TRAIN, split="train"),
    processing_class=tokenizer,
    peft_config=LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05,
                           target_modules="all-linear", task_type="CAUSAL_LM"),
    args=SFTConfig(output_dir=f"{OUT}/checkpoints", num_train_epochs=3,
                   per_device_train_batch_size=4, learning_rate=2e-4, fp16=True,
                   logging_steps=5, save_strategy="no", report_to="none", seed=42),
)
for p in trainer.model.parameters():
    if p.requires_grad:
        p.data = p.data.float()
trainer.train()
trainer.save_model(f"{OUT}/adapter")


Step,Training Loss
5,0.329391
10,0.066363
15,0.016378
20,0.015340
25,0.005130
30,0.003331
35,0.002292


In [4]:
# Cell 4: predict the smoke questions with and without the adapter
model = trainer.model.eval()
prompts = [json.loads(line) for line in open(SMOKE)]


def generate(messages):
    inputs = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, return_dict=True, return_tensors="pt").to(model.device)
    out = model.generate(**inputs, max_new_tokens=256, do_sample=False)
    text = tokenizer.decode(out[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    match = re.search(r"```(?:sql)?\s*(.*?)```", text, re.DOTALL)  # same as strip_fences
    return (match.group(1) if match else text).strip().rstrip(";")


def predict(name):
    with open(f"{OUT}/{name}.jsonl", "w") as f:
        for p in prompts:
            start = time.perf_counter()
            sql = generate(p["prompt"])
            latency_ms = round((time.perf_counter() - start) * 1000)
            f.write(json.dumps({"id": p["id"], "pred_sql": sql, "latency_ms": latency_ms}) + "\n")
            print(p["id"], sql)


with model.disable_adapter():
    predict("qwen2.5-coder-1.5b-base")
predict("qwen2.5-coder-1.5b-qlora-toy")

smoke_01 SELECT COUNT(*) FROM merchants WHERE city = 'Jakarta'
smoke_02 SELECT COUNT(*) FROM transactions WHERE status = 'failed' AND created_at >= '2026-09-30'
smoke_03 SELECT SUM(amount_idr) AS total_gmv FROM transactions WHERE status = 'success' AND created_at >= '2026-09-30'
smoke_04 SELECT name FROM banks WHERE type = 'ewallet'
smoke_05 SELECT 
  m.category AS category,
  SUM(t.amount_idr) AS gmv
FROM 
  merchants m
JOIN 
  transactions t ON m.merchant_id = t.merchant_id
WHERE 
  t.created_at >= '2026-09-30'
GROUP BY 
  m.category
smoke_06 SELECT 
  method, 
  COUNT(*) AS success_count, 
  CAST(COUNT(*) AS REAL) * 100 / SUM(CASE WHEN status = 'success' THEN 1 ELSE 0 END) AS success_rate
FROM 
  transactions
WHERE 
  created_at >= DATE '2026-09-30'
GROUP BY 
  method
smoke_07 SELECT T1.city, COUNT(*) FROM transactions AS T1 JOIN merchants AS T2 ON T1.merchant_id = T2.merchant_id WHERE T1.created_at >= '2026-09-30' GROUP BY T1.city
smoke_08 SELECT COUNT(*) FROM transactions WHERE cr